# Evidence RAG v2

This version uses all-MiniLM-L6-v2 from Hugging Face as the embedding model and Gemini 3.1 Flash Lite as the inference model. This model is available through the Gemini API and allows for a limited amount of use in the free tier.

In [ ]:
# Install and import libraries

%pip install requests beautifulsoup4 pdfminer.six lxml sentence-transformers
%pip install selenium google-colab-selenium

import requests
import re
import io
import os
import time
import pickle
import torch
import numpy as np
import google_colab_selenium as gs
from google.colab import drive
from bs4 import BeautifulSoup
from pdfminer.high_level import extract_text
from urllib.parse import urljoin, urlparse
from pathlib import Path
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from sentence_transformers import SentenceTransformer

In [ ]:
# Gemini API setup

import google.generativeai as genai
from google.colab import userdata
api_key = userdata.get("GEMINI_API_KEY")

In [ ]:
# Load data from storage

fetch_new = True
time.sleep(1)
to_load = input("Would you like to load pre-saved chatbot data? (y/n) ")
try:
    if to_load.lower()[0] == "y":
        drive.mount('/content/drive')
        with open('/content/drive/MyDrive/rag_corpus.pkl', 'rb') as f:
            corpus = pickle.load(f)
        with open('/content/drive/MyDrive/rag_documents.pkl', 'rb') as f:
            documents = pickle.load(f)
        with open('/content/drive/MyDrive/rag_papers.pkl', 'rb') as f:
            papers = pickle.load(f)
        with open('/content/drive/MyDrive/rag_vectors.pkl', 'rb') as f:
            vectors = pickle.load(f)
        print("\nLoaded", len(documents), "guidelines,", len(papers), "review articles,", len(corpus), "chunks and", len(vectors), "vectors")
        fetch_new = False
except Exception as e:
    raise(e)

Would you like to load pre-saved chatbot data? (y/n) y
Mounted at /content/drive

Loaded 880 guidelines, 1003 review articles, 60201 chunks and 60201 vectors


In [ ]:
# Guideline sites and review journals

sites = [
    "https://www.rch.org.au/clinicalguide",
    "https://eyeandear.org.au/health-professionals/clinical-practice-guidelines/",
    "https://dermnetnz.org/topics",
    "https://sti.guidelines.org.au",
    "https://www.cdc.gov/yellow-book/hcp/contents/index.html",
    "https://www.thewomens.org.au/health-professionals/clinical-resources/clinical-guidelines-gps"
]

journals = [
    "Aust J Gen Pract",
    "Am Fam Physician",
    "Dtsch Arztebl Int",
    "Can Fam Physician",
    "CMAJ",
    "Aust Prescr",
    "Postgrad Med"
]

In [ ]:
# PubMed search terms

journal_query = " OR ".join([f'"{j}"[jour]' for j in journals])

query = """
Review[pt]
AND free full text[sb]
AND "last 5 years"[pdat]
AND English[lang]
AND
(
""" + journal_query + "\n)"

In [ ]:
# Crawl guideline websites

def detect_content_type(url):
    try:
        r = requests.get(url, stream=True, timeout=10)
        content_type = r.headers.get("Content-Type", "").lower()
        content_disp = r.headers.get("Content-Disposition", "").lower()
        if "pdf" in content_type:
            return "pdf"
        if "app.prompt.org.au" in url:
            return "pdf_link"
        if "attachment" in content_disp and "pdf" in content_disp:
            return "pdf"
        if "download" in url:
            return "pdf"
        return "html"
    except:
        r = requests.head(url, allow_redirects=True)
        content_type = r.headers.get("Content-Type", "")
        if "pdf" in content_type.lower():
            return "pdf"
        return "html"

def extract_html_text(url):
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "lxml")
    if (use_driver or len(soup) < 3):
        driver.get(url)
        try:
            element = WebDriverWait(driver, 10).until(
                EC.none_of(
                    EC.title_contains("Checking your browser"),
                    EC.title_contains("Just a moment")
                )
            )
            time.sleep(1)
        except:
            pass
        soup = BeautifulSoup(driver.page_source, "lxml")
    title = soup.title.text.strip('\r\n\t ') if soup.title else ""
    if title[:12] == "RACGP - AJGP":
        title = soup.find("h1").text.strip('\r\n\t ') + " - AJGP"
    sections = []
    for heading in soup.find_all(["h1", "h2", "h3", "h4", "h5", "strong"]):
        heading_text = heading.get_text(strip=True)
        content_parts = []
        for sibling in [*heading.next_siblings, *heading.parent.next_siblings]:
            if getattr(sibling, "name", None) in ["h1", "h2", "h3", "h4", "h5", "strong"]:
                break
            if sibling.name in ["p", "li"]:
                content_parts.append(sibling.get_text(strip=True))
            # handle nested lists
            if sibling.name == "ul":
                for li in sibling.find_all("li"):
                    content_parts.append(li.get_text(strip=True))
        text = " ".join(content_parts)
        if len(text) > 20:  # filter tiny sections
            sections.append({
                "heading": heading_text,
                "text": text
            })
    return title, sections

def extract_pdf_text(url):
    r = requests.get(url, allow_redirects=True)
    file_like = io.BytesIO(r.content)
    text = extract_text(file_like)
    sections = []
    current = {"heading": "Document", "text": ""}
    for line in text.split("\n"):
        line = line.strip()
        # detect headings (simple heuristic)
        if line.isupper() and len(line) < 80:
            if current["text"]:
                sections.append(current)
            current = {"heading": line, "text": ""}
        else:
            current["text"] += " " + line
    if current["text"]:
        sections.append(current)
    title = sections[0]["heading"] if sections else ""
    return title, sections

def file_exists_in_folder(path):
    # Custom condition: returns True if the folder is not empty
    def _predicate(driver):
        files = os.listdir(path)
        valid_files = [f for f in files if f.endswith('.pdf')]
        return len(valid_files) > 0
    return _predicate

def extract_downloaded_pdf_text(url):
    driver.get(url)

    # Wait for download (adjust based on file size)
    try:
        WebDriverWait(driver, 10).until(file_exists_in_folder(download_path))
        print("Downloaded", os.listdir(download_path))
    except Exception as e:
        print("Download timed out or failed.")

    # Get all files in the directory with their full paths
    files = [os.path.join(download_path, f) for f in os.listdir(download_path)]
    # Filter to include only files (ignoring subdirectories)
    files = [f for f in files if os.path.isfile(f)]
    pdf_files = [f for f in files if ".pdf" in f]
    title = ""

    if pdf_files:
        # Find the file with the maximum modification time (most recent)
        latest_file = max(pdf_files, key=os.path.getmtime)
        title = Path(latest_file).stem

        # Extract and process text from file
        text = extract_text(latest_file)
        sections = []
        current = {"heading": "Document", "text": ""}
        for line in text.split("\n"):
            line = line.strip()
            # detect headings (simple heuristic)
            if line.isupper() and len(line) < 80:
                if current["text"]:
                    sections.append(current)
                current = {"heading": line, "text": ""}
            else:
                current["text"] += " " + line
        if current["text"]:
            sections.append(current)

        # Delete the file
        for file in files:
            os.remove(file)
        return title, sections

    else:
        print("Unable to locate file")
        return None

def discover_links(base_url):
    global use_driver
    use_driver = False
    try:
        r = requests.get(base_url, timeout=(10,10), headers=headers)
        soup = BeautifulSoup(r.text, "lxml")
    except:
        driver.get(base_url)
        soup = BeautifulSoup(driver.page_source, "lxml")
    if len(soup.find_all("a", href=True)) == 0:
        driver.get(base_url)
        soup = BeautifulSoup(driver.page_source, "lxml")
        use_driver = True
    base_domain = urlparse(base_url).netloc
    subs = [
        ("rch.org.au", "rch.org.au/clinicalguide/guideline_index"),
        ("dermnetnz.org", "dermnetnz.org/topics"),
        ("cdc.gov", "cdc.gov/yellow-book/hcp")
    ]
    for dom, sub in subs:
        if dom in base_domain:
            base_domain = sub
    links = []
    for a in soup.find_all("a", href=True):
        url = urljoin(base_url, a["href"])
        domain = urlparse(url).netloc
        if (
            (base_domain in url
               and not "eyeandear.org.au" in domain
               and not "thewomens.org.au" in domain)
            or "prompt.org.au" in domain
            or "/download/" in url
            or url.endswith(".pdf")
        ):
            links.append(url)
    return list(set(links))

def extract_document(url):
    doc_type = detect_content_type(url)
    try:
        if doc_type == "pdf":
            t, s = extract_pdf_text(url)
        elif doc_type == "pdf_link":
            t, s = extract_downloaded_pdf_text(url)
        else:
            t, s = extract_html_text(url)
        return {
            "url": url,
            "title": t,
            "sections": s
        }
    except Exception as e:
        print("Couldn't extract", url)
        print(e)
        return None

if fetch_new:
    documents = []

    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    use_driver = False

    download_path = "/content/temp_downloads"
    if not os.path.exists(download_path):
        os.makedirs(download_path)
    files = [os.path.join(download_path, f) for f in os.listdir(download_path)]
    files = [f for f in files if os.path.isfile(f)]
    for file in files:
        print("Deleting", file)
        os.remove(file)

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    # Set Chrome preferences for automatic downloads
    prefs = {
        "download.default_directory": download_path,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "plugins.always_open_pdf_externally": True  # Bypasses the PDF viewer
    }
    options.add_experimental_option("prefs", prefs)
    options.page_load_strategy = 'eager'

    driver = gs.Chrome(options=options)

    for site in sites:
        print("\nCrawling", site)
        links = discover_links(site)
        for link in links:
            print(len(documents), end=": ")
            print("Extracting", link)
            doc = extract_document(link)
            if doc:
                documents.append(doc)

    driver.quit()
    print("\nExtracted", len(documents), "documents in total")

In [ ]:
# Search PubMed

BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
max_results = 1000
max_extract = 1000

def search_pubmed(query, max_results=max_results):
    url = f"{BASE}/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json"
    }
    r = requests.get(url, params=params).json()
    results = r["esearchresult"]["idlist"]
    print(len(results), "results found in PubMed search")
    return results

def get_pmcid(pmid, retries=3):
    url = "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"
    params = {
        "ids": pmid,
        "format": "json"
    }
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=10)
            data = r.json()
            records = data.get("records", [])
            if records and "pmcid" in records[0]:
                return records[0]["pmcid"]
            return None
        except Exception as e:
            print(f" (get_pmcid attempt {attempt+1} failed: {e})", end="")
            time.sleep(2 ** attempt)  # exponential backoff: 1s, 2s, 4s
    return None

def download_pmc_xml(pmcid):
    url = f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmcid}/?format=xml"
    r = requests.get(url)
    return r.text

def extract_pmc_text(xml):
    soup = BeautifulSoup(xml, "lxml")
    title = soup.title.text.strip('\r\n\t ')
    sections = []
    for sec in soup.find_all("sec"):
        heading = sec.find("title")
        heading_text = heading.get_text() if heading else "Section"
        paragraphs = sec.find_all("p")
        text = " ".join(p.get_text() for p in paragraphs)
        if text.strip():
            sections.append({
                "heading": heading_text,
                "text": text
            })
    return title, sections

def get_article_url(pmid):
    params = {
        "db": "pubmed",
        "id": pmid,
        "retmode": "xml"
    }
    r = requests.get(BASE + "/efetch.fcgi", params=params)
    soup = BeautifulSoup(r.text, "xml")
    article_ids = soup.find_all("ArticleId")
    for aid in article_ids:
        if aid.get("IdType") == "doi":
            return "https://doi.org/" + aid.text
    return None

if fetch_new:
    papers = []
    pmids = search_pubmed(query)
    driver = gs.Chrome(options=options)

    for pmid in pmids:
        pmcid = get_pmcid(pmid)
        print("Extracting PMID", pmid, end="")
        if pmcid:
            url = "https://pmc.ncbi.nlm.nih.gov/articles/" + pmcid
            title, sections = extract_html_text(url)
            if not sections:
                use_driver = True
                title, sections = extract_html_text(url)
                use_driver = False
            print(":", title)
        else:
            url = get_article_url(pmid)
            if url:
                title, sections = extract_html_text(url)
                if not sections:
                    use_driver = True
                    title, sections = extract_html_text(url)
                    use_driver = False
                print(":", title)
            else:
                text = ""
                title = ""
                sections = []
                print(" - no PMCID")
                print("")
        if (sections and title != "Error: DOI Not Found" and title != "Just a moment..."):
            papers.append({
                "pmid": pmid,
                "url": url,
                "title": title,
                "sections": sections
            })
        else:
            print("  Couldn't extract PMID", pmid)
        if len(papers) >= max_extract:
            break

    driver.quit()
    print("\nExtracted", len(papers), "papers in total")

In [ ]:
# Extract Cochrane Reviews

query_cr = """
Review[pt]
AND free full text[sb]
AND "last 5 years"[pdat]
AND (therapy[Majr] OR diagnosis[Majr])
AND "Cochrane Database Syst Rev"[jour]
"""

max_cr_extract = 500

def extract_cochrane_review(url):
    r = requests.get(url)
    soup = BeautifulSoup(r.text, "lxml")
    if use_driver:
        driver.get(url)
        try:
            element = WebDriverWait(driver, 10).until(
                EC.none_of(
                    EC.title_contains("Checking your browser"),
                    EC.title_contains("Just a moment")
                )
            )
            time.sleep(1)
        except:
            pass
        soup = BeautifulSoup(driver.page_source, "lxml")
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else ""
    pls_text = []
    # find headings that match PLS
    for heading in soup.find_all(["h2", "h3"]):
        h_text = heading.get_text(strip=True).lower()
        if "plain language" in h_text:
            # collect content until next heading
            for sib in heading.next_siblings:
                if getattr(sib, "name", None) in ["h2", "h3"]:
                    break
                if sib.name == "p":
                    pls_text.append(sib.get_text(strip=True))
                if sib.name == "div":
                    pls_text.extend(
                        p.get_text(strip=True)
                        for p in sib.find_all("p")
                    )
            break  # only need first match
    text = " ".join(pls_text)
    sections = {
        "heading": "Plain language summary",
        "text": text
    }
    return title, sections

if fetch_new:
    max_extract = len(papers) + max_cr_extract
    pmids = search_pubmed(query_cr)
    print("\nExtracting Cochrane Reviews\n")
    driver = gs.Chrome(options=options)

    for pmid in pmids:
        pmcid = get_pmcid(pmid)
        print("Extracting PMID", pmid, end="")
        if pmcid:
            url = "https://pmc.ncbi.nlm.nih.gov/articles/" + pmcid
            title, sections = extract_cochrane_review(url)
            if not sections["text"]:
                use_driver = True
                title, sections = extract_cochrane_review(url)
                use_driver = False
            print(":", title)
        else:
            print(" - no PMCID")
            continue
        if sections["text"]:
            papers.append({
                "pmid": pmid,
                "url": url,
                "title": title,
                "sections": sections
            })
        else:
            print("  Couldn't extract PMID", pmid)
        if len(papers) >= max_extract:
            break

    driver.quit()
    print("\nExtracted", len(papers), "papers in total")

In [ ]:
# Embedding setup

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_prog(texts):
    return np.array(embed_model.encode(texts, show_progress_bar=True))

def embed(texts):
    return np.array(embed_model.encode(texts, show_progress_bar=False))

def format_chunk(chunk):
    title = chunk["title"]
    heading = chunk["heading"]
    return f"""
Title: {title}
Section: {heading}
Content:
{chunk['text']}
""".strip()

In [ ]:
# Chunking and embedding

def chunk(doc, chunk_size=1000, overlap=100):
    chunks = []
    for section in doc["sections"]:
        try:
            text = section["text"]
            i = 0
            while i < len(text):
                chunk_text = text[i:i+chunk_size]
                chunks.append({
                    "text": chunk_text,
                    "title": doc["title"],
                    "heading": section["heading"],
                    "source": doc["url"]
                })
                i += chunk_size - overlap
        except:
            continue
    return chunks

if fetch_new:
    corpus = []

    for paper in papers:
        for c in chunk(paper):
            corpus.append(c)

    for page in documents:
        for c in chunk(page):
            corpus.append(c)

    print(len(corpus), "chunks obtained")

    texts = [format_chunk(c) for c in corpus]

    vectors = embed_prog(texts)

In [ ]:
# Retrieval

def search(query, k=10, threshold=0.4):
    q = embed([query])[0]
    sims = vectors @ q / (
        np.linalg.norm(vectors, axis=1) * np.linalg.norm(q)
    )
    idx = np.argsort(-sims)
    results = []
    for i in idx:
        if sims[i] < threshold:
            continue
        results.append((corpus[i], sims[i]))
        if len(results) >= k:
            break
    return results

In [ ]:
# Query functions

def ask(question):
    results = search(question, threshold=0.3)
    if len(results) == 0:
        print("\nNo relevant sources found")
        return None
    context = "\n\n".join(format_chunk(r[0]) for r in results)
    prompt = f"""

Use the medical evidence below to answer. Construct an answer as a short sentence or list of points. Don't just quote.

Context:
{context}

Question:
{question}
"""

    response = gemini.generate_content(prompt)
    print("\n" + response.text)
    print("\nSources:")
    sources = []
    for r in results:
        d, sim = r
        if d["source"] not in sources:
            sources.append(d["source"])
            print(d["source"], end="")
            if d["title"]:
                print(":", d["title"])
            else:
                print("")

In [ ]:
# Save chatbot data to Drive

if fetch_new:
    to_save = input("Would you like to save the database to your Drive? (y/n) ")
    try:
        if to_save.lower()[0] == "y":
            drive.mount('/content/drive')
            with open('/content/drive/MyDrive/rag_corpus.pkl', 'xb') as f:
                pickle.dump(corpus, f)
            with open('/content/drive/MyDrive/rag_documents.pkl', 'xb') as f:
                pickle.dump(documents, f)
            with open('/content/drive/MyDrive/rag_papers.pkl', 'xb') as f:
                pickle.dump(papers, f)
            with open('/content/drive/MyDrive/rag_vectors.pkl', 'xb') as f:
                pickle.dump(vectors, f)
            print("\nSaved", len(documents), "guidelines,", len(papers), "review articles,", len(corpus), "chunks and", len(vectors), "vectors")
    except FileExistsError:
        print("\nError: The file already exists. Please manually delete or rename files if you still wish to save data.")
    except Exception as e:
        print(e)

## Using Gemini 3.1 Flash Lite

In [ ]:
# Model setup

genai.configure(api_key=api_key)
gemini = genai.GenerativeModel("gemini-3.1-flash-lite")

In [ ]:
# User query

print("Answers given by this chatbot are AI-generated and MAY CONTAIN ERRORS.")
print("Please check the provided sources.")

while True:
    question = input("\n\nAsk a question: ")
    if question.lower() == "break":
        break
    ask(question)

Answers given by this chatbot are AI-generated and MAY CONTAIN ERRORS.
Please check the provided sources.


Ask a question: How do you treat febrile convulsions?

Treatment for febrile seizures focuses on the following:

*   **Seizure management:** If the seizure lasts longer than 5 minutes, proceed with acute seizure management protocols.
*   **Fever management:** Address the underlying cause of the fever.
*   **Medication:** Ongoing treatment with antiepileptic drugs is generally not recommended.
*   **Clinical assessment:** Simple febrile seizures do not require additional investigations; however, if the seizure is complex or there are concerns for serious infection (e.g., sepsis, meningitis), consult a pediatric team and consider targeted testing or CNS imaging.

Sources:
https://www.rch.org.au/clinicalguide/guideline_index/Febrile_seizure/: Clinical Practice Guidelines : Febrile seizure
https://www.rch.org.au/clinicalguide/guideline_index/Anticonvulsant_poisoning/: Clinical Practi

ERROR:tornado.access:503 POST /v1beta/models/gemini-3.1-flash-lite-preview:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3654.32ms



An earache can be caused by conditions originating within the ear or by referred pain from elsewhere in the body:

*   **Infection of the ear canal (Otitis Externa):** Occurs when the protective barriers of the ear canal break down, often due to moisture, trauma, or foreign objects, leading to bacterial or fungal overgrowth.
*   **Dental Pathology:** The most common cause of non-ear-related (referred) otalgia, resulting from shared nerve pathways (CN V) between the teeth, oral mucosa, and the ear.
*   **Upper Airway Inflammation or Infection:** Conditions like tonsillitis, pharyngitis, or sinusitis can cause referred pain through shared neural pathways involving the glossopharyngeal and vagus nerves.
*   **Laryngopharyngeal Reflux (LPR):** The backflow of stomach acid into the upper airway irritates surrounding tissues and nerves, which can be felt as ear pain.
*   **Oral Mucosal Lesions:** Conditions such as aphthous ulcers or malignancy can manifest as ear pain due to shared sensory